# proteinCAD — GPU worker on Colab

This notebook turns a Colab session into the compute side of proteinCAD. The app
stays where it is (your laptop, or your website); this notebook runs the models
and hands results back.

There are two design stages, and they are asked for separately from the app:

1. **Backbone.** RFdiffusion draws a new chain against the residues you picked.
   Minutes. This is a shape — every residue in it is glycine, because nothing
   has chosen the amino acids yet. (ESM3-open is an optional second engine for
   this stage — section 7 — and it writes a sequence as it goes, so its output
   is a protein rather than a shape. The check below is still worth running on
   it, because it is an independent model's opinion of that sequence.)
2. **Sequence and fold.** ProteinMPNN chooses those amino acids against the
   target, and ESMFold then folds the sequence on its own to see whether it
   comes back to the shape it was designed for. This is the step that says
   whether stage 1 produced a protein or only a picture of one.

Work through the notebook in two stages of its own:

1. **Prove the connection** with the `echo` generator. No GPU, no model, about a
   minute. If this works, everything between the viewer and Colab is correct.
2. **Install the models.** Same worker, same URL, real designs.

Runtime → Change runtime type → **T4 GPU** (or better, if folding) before
stage 2.

## 1. Get the worker onto the machine

Run the cell and pick `proteincad/colab_worker.py` from your checkout. (Once the
repository is public, replace this with a `git clone` — see the commented line.)

In [ ]:
from google.colab import files
uploaded = files.upload()          # choose proteincad/colab_worker.py
WORKER = next(iter(uploaded))
print("worker:", WORKER)

# Once the repo is public this is all you need instead:
# !git clone --depth 1 https://github.com/<you>/proteinCAD.git
# WORKER = "proteinCAD/proteincad/colab_worker.py"

## 2. Open a tunnel

Colab has no public address, so we borrow one. `cloudflared` needs no account
and no signup; the URL lasts as long as this session.

In [ ]:
import re, subprocess, time, os, secrets, threading, collections, queue

PORT = 8000
TOKEN = secrets.token_urlsafe(16)      # shared secret; paste it into the app too

!wget -q -c https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

tunnel = None
tunnel_output = collections.deque(maxlen=300)


def start_tunnel(port=PORT, timeout=90):
    """Open a public URL for the worker.

    cloudflared logs continuously. Its output *must* keep being read: if the
    pipe buffer fills, cloudflared blocks on write, stops servicing the tunnel,
    and the hostname quietly stops resolving. So a thread drains it for as long
    as it runs.
    """
    global tunnel
    if tunnel and tunnel.poll() is None:
        tunnel.terminate()
        try:
            tunnel.wait(timeout=5)
        except subprocess.TimeoutExpired:
            tunnel.kill()

    tunnel_output.clear()
    found = queue.Queue()
    tunnel = subprocess.Popen(
        ["./cloudflared", "tunnel", "--url", f"http://localhost:{port}",
         "--no-autoupdate", "--loglevel", "info"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )

    def drain():
        for line in tunnel.stdout:          # runs until cloudflared exits
            tunnel_output.append(line.rstrip())
            match = re.search(r"https://[-\w.]+\.trycloudflare\.com", line)
            if match:
                found.put(match.group(0))

    threading.Thread(target=drain, daemon=True).start()

    try:
        return found.get(timeout=timeout)
    except queue.Empty:
        print("*** no tunnel URL appeared ***")
        print("\n".join(list(tunnel_output)[-15:]))
        return None


PUBLIC_URL = start_tunnel()
print("\npublic URL:", PUBLIC_URL)
print("token:     ", TOKEN)

## 3. Start the worker (echo — connection test)

This keeps running in the background. The `echo` generator sends the target back
as a "design", which is enough to prove the round trip.

In [ ]:
import subprocess, os, sys, time, json, socket, urllib.request, threading, collections

env = dict(os.environ, PROTEINCAD_WORKER_TOKEN=TOKEN)
worker = None
worker_output = collections.deque(maxlen=200)


def port_busy(port):
    with socket.socket() as probe:
        probe.settimeout(0.3)
        return probe.connect_ex(("127.0.0.1", port)) == 0


def stop_worker(port=None, quiet=False):
    """Free the port, so re-running a cell replaces the worker instead of
    colliding with it. Asks politely first, then insists."""
    port = port or PORT
    global worker
    if worker and worker.poll() is None:
        worker.terminate()
        try:
            worker.wait(timeout=5)
        except subprocess.TimeoutExpired:
            worker.kill()

    if port_busy(port):
        try:  # an older worker we no longer have a handle on
            request = urllib.request.Request(
                f"http://localhost:{port}/shutdown", data=b"{}", method="POST",
                headers={"Content-Type": "application/json", "Authorization": f"Bearer {TOKEN}"})
            urllib.request.urlopen(request, timeout=3)
        except Exception:
            pass
    if port_busy(port):
        for command in (["fuser", "-k", f"{port}/tcp"], ["pkill", "-f", f"--port {port}"]):
            try:
                subprocess.run(command, capture_output=True, timeout=5)
            except Exception:
                pass

    for _ in range(20):
        if not port_busy(port):
            return True
        time.sleep(0.25)
    if not quiet:
        print(f"! port {port} is still held by something. Try: !fuser -k {port}/tcp")
    return False


def start_worker(*args):
    """Start the worker, capture its output, and confirm it is actually up.

    A worker that dies on startup leaves the tunnel with nothing behind it, and
    the app then reports an unhelpful gateway error. Checking here turns that
    into the real message.
    """
    global worker
    stop_worker(quiet=True)
    worker_output.clear()
    worker = subprocess.Popen(
        [sys.executable, "-u", WORKER, "--port", str(PORT), *args],
        env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    threading.Thread(
        target=lambda: [worker_output.append(line.rstrip()) or print(line, end="")
                        for line in worker.stdout],
        daemon=True,
    ).start()

    for _ in range(20):
        time.sleep(0.5)
        if worker.poll() is not None:
            lines = [line for line in worker_output if line.strip()]
            print("\n*** the worker exited immediately ***")
            if lines:
                print("reason:", lines[-1])
            return None
        try:
            return json.load(urllib.request.urlopen(f"http://localhost:{PORT}/health", timeout=2))
        except Exception:
            continue
    print("\n*** the worker did not answer /health — output is above ***")
    return None


def wait_for_preflight(timeout=180):
    """Wait for the worker's environment check and return it.

    The worker answers /health straight away and probes its environment in a
    background thread -- importing torch, dgl and RFdiffusion takes the better
    part of a minute, and a liveness check that blocks for that long makes a
    healthy worker look dead. So /health carries `preflight: null` until the
    check finishes.
    """
    import urllib.request, json, time
    end = time.time() + timeout
    while time.time() < end:
        try:
            with urllib.request.urlopen(f"http://localhost:{PORT}/health", timeout=5) as response:
                flight = json.load(response).get("preflight")
                if flight:
                    return flight
        except Exception:
            pass
        time.sleep(2)
    print("the preflight has not finished yet — check the worker output below")
    return None


health = start_worker("--generator", "echo")
print("\nworker health:", health)

In [ ]:
def check_link():
    """Test the path the app actually uses: through the tunnel, to the worker.

    Run this whenever a job says the endpoint is unreachable. It separates
    "tunnel is gone" from "worker is down" in one step.
    """
    import urllib.request, urllib.error, socket
    if tunnel and tunnel.poll() is not None:
        print("cloudflared has exited — run start_tunnel() again, and restart the app")
        print("\n".join(list(tunnel_output)[-10:]))
        return False
    if not port_busy(PORT):
        print(f"nothing is listening on {PORT} — run the worker cell")
        return False
    try:
        with urllib.request.urlopen(f"{PUBLIC_URL}/health", timeout=20) as response:
            print("link ok:", response.read().decode()[:200])
            return True
    except urllib.error.HTTPError as error:
        print(f"the tunnel answered {error.code} — worker not reachable behind it")
    except Exception as error:
        reason = getattr(error, "reason", error)
        if isinstance(reason, socket.gaierror):
            print("the tunnel hostname no longer resolves — it has closed.")
            print("Run start_tunnel(), then restart proteinCAD with the new URL.")
        else:
            print("link failed:", reason)
    return False


check_link()

## 4. Point the app at it

On the machine running proteinCAD, restart it with the URL and token this
notebook printed:

```bash
PROTEINCAD_COMPUTE_URL=<public URL> \
PROTEINCAD_COMPUTE_TOKEN=<token> \
python3 -m proteincad
```

or equivalently:

```bash
python3 -m proteincad --compute-url <public URL> --compute-token <token>
```

The Design tab's **on** menu will now offer `remote` alongside `mock`. Pick
`remote`, press **Run design**, and the job goes to this notebook.

Nothing about the URL is written into the app — it is configuration, so the same
build works against Colab now and against AWS later by changing one variable.

## 5. Install the models

One cell. It clones RFdiffusion and ProteinMPNN, fetches the model weights,
installs the dependencies, and then proves the result by importing everything in
the interpreter that will run it. Re-running is safe and cheap — it only does
what is still missing.

**Expect fifteen minutes and roughly 13 GB**: about 4 GB of torch and
RFdiffusion checkpoints, and about 8.5 GB of ESMFold weights. Progress streams
into the cell; silence for a minute or two during a large download is normal.

Set **Runtime → Change runtime type → T4 GPU** before running this.

If you only want backbones for now, `--only rfdiffusion` skips the folding half
and its 8.5 GB. `--only fold` adds it later. `--skip-weights` installs the
folding code but leaves the weights to download on the first fold job, which is
the same total spent somewhere less obvious.

RFdiffusion has eight checkpoints and chooses between them from the job itself.
This fetches the two a binder run picks between; anything else — motif
inpainting, fold conditioning, the active-site model — is downloaded by the
first job that needs it, 484 MB, once. `--weights all` fetches all eight now
(3.9 GB) if you would rather wait here than mid-run.

### Why this is not just `pip install rfdiffusion dgl`

Three traps, all of which this cell handles, and all of which produce errors
that point somewhere other than the cause:

* **DGL is not usefully on PyPI.** The newest release there has no wheel for
  current Pythons, so pip builds an ancient version from source and the failure
  surfaces as `cannot import name 'Mapping' from 'collections'` — which looks
  like a Python problem. PyPI also only ever carried the CPU build, which cannot
  run the model. The real wheels are on DGL's own index.
* **That index lists wheels it will not serve.** Some objects come back `403
  AccessDenied` while their neighbours download fine, so every candidate is
  checked before pip is pointed at it.
* **A DGL wheel is built against one exact torch.** It loads libraries named for
  that build — `libdgl_sparse_pytorch_2.6.0.so` — so torch 2.6.1 resolves to the
  same index and then fails at import with nothing to suggest why.

ProteinMPNN and ESMFold have less of this trouble: ProteinMPNN keeps its weights
in the checkout, and ESMFold is reached through `transformers`, which is plain
torch — no second CUDA stack beside the one RFdiffusion needs. One pin,
`transformers<5`, to keep the import surface small: ESMFold is a quiet corner of
that library and 5.x rebuilt the machinery underneath it, so the same import now
also passes through four newer lazy modules.

There is a fourth trap, and it is one this cell creates. DGL forces an exact
torch, and Colab's `torchvision` was compiled against the torch Colab shipped
with. Once torch moves, torchvision can no longer register its own operators —
and because importing `transformers` reaches for torchvision, loading a protein
folder fails with `RuntimeError: operator torchvision::nms does not exist`. This
cell checks torch's compiled siblings after pinning it and reinstalls any that
the move stranded.

In [ ]:
import os, sys

# The same interpreter start_worker() uses, so the packages land where the
# models will look for them. `python` on PATH is not reliably that one.
os.environ["PY"], os.environ["WORKER"] = sys.executable, WORKER

# Prints a per-stage report at the end; anything it could not fix is named there
# with the reason. Safe to re-run -- it only does what is still missing.
#
# Add --only rfdiffusion to skip the folding half and its 8.5 GB for now.
# Add --weights all to fetch every RFdiffusion checkpoint now rather than on
# the first job that needs one.
!"$PY" "$WORKER" setup --rfdiffusion /content/RFdiffusion --proteinmpnn /content/ProteinMPNN

## 6. Restart the worker with the models

Same port, same tunnel, same token — only the generator changes, so the app needs
no attention. Do not re-run the tunnel cell: that would hand you a new URL.

One worker serves both stages. Each checks its own environment, so a machine
with RFdiffusion but no folding weights runs backbone jobs normally and says
exactly what is missing when asked for the rest.

In [ ]:
health = start_worker("--generator", "rfdiffusion",
                      "--rfdiffusion", "/content/RFdiffusion",
                      "--proteinmpnn", "/content/ProteinMPNN")

# The worker checks its environment in a background thread, so /health answers at
# once and the preflight lands a little later. Wait for it rather than guessing.
flight = wait_for_preflight()
if flight:
    stages = flight.get("stages", {})
    print("\ninterpreter:", stages.get("binder", {}).get("interpreter"),
          "| python", flight.get("python"))
    print("torch      :", flight.get("torch"), "| cuda:", flight.get("cuda"), flight.get("gpu", ""))
    print("dgl        :", flight.get("dgl"))
    print("weights    :", ", ".join(flight.get("weights") or []) or "NONE FOUND")

    fold = stages.get("fold", {})
    print("sequences  :", fold.get("proteinmpnn"),
          "--", (fold["problems"][0] if fold.get("problems") else "ok"))
    print("folding    :", fold.get("folder"),
          "| transformers", fold.get("versions", {}).get("transformers") or "not installed",
          "--", (fold["folding"][0] if fold.get("folding") else "ok"))

    for problem in flight.get("problems", []):
        print("  !", problem)
    if not flight.get("problems"):
        print("\nReady. Run a design from the app; progress appears below.")
    # Folding is not load-bearing: without it the second stage still returns
    # sequences, just without the check that they fold.
    if fold.get("folding"):
        print("\nSequences will work; the folding check will not. To repair just that half:")
        print('  !"$PY" "$WORKER" setup --only fold')

## 7. Optional: add ESM3-open as a second engine

RFdiffusion draws a shape and the second stage gives it a sequence. **ESM3**
writes both at once, and takes a different kind of instruction: masked tracks
(sequence, structure, secondary structure, exposure, function) and a plan saying
which order to fill them in. In the app it is the `with` menu in the Design
panel.

The weights are public — 5.5 GB, no account and no token needed. They are not
part of the setup cell above because a session using only RFdiffusion has no use
for them.

Skip this section entirely if you only want RFdiffusion. Everything above keeps
working.

In [ ]:
import os, sys

os.environ["PY"], os.environ["WORKER"] = sys.executable, WORKER
# 5.5 GB, once per session. --only esm3 is separate from `all` on purpose: it is
# a second engine rather than a missing piece of the first.
!"$PY" "$WORKER" setup --only esm3

In [ ]:
# Both engines, one worker, same port and tunnel -- so the app needs no
# attention. Which engine a design uses is chosen per job in the panel.
health = start_worker("--generator", "rfdiffusion,esm3",
                      "--rfdiffusion", "/content/RFdiffusion",
                      "--proteinmpnn", "/content/ProteinMPNN")

flight = wait_for_preflight()
if flight:
    stages = flight.get("stages", {})
    print("\nengines   :", ", ".join(sorted(stages)) or "none reported")
    for name in ("binder", "esm3", "fold"):
        stage = stages.get(name)
        if not stage:
            continue
        trouble = stage.get("problems") or []
        print(f"  {name:9s}", "ok" if not trouble else trouble[0][:90])

In [ ]:
# Everything the worker has said recently. It is already streaming into the cell
# above; this is for looking back after the fact.
print("\n".join(worker_output))

## 8. Using the two stages from the app

In the Design tab:

1. **Pick site** → click residues on the target.
2. **Run design** → RFdiffusion draws backbones against them. They arrive
   already positioned on the residues you picked.

   `make` above it picks the protocol: binder design, motif scaffolding,
   unconditional monomers, symmetric oligomers, partial diffusion, fold
   conditioning. The last three need no target at all, or a structure rather
   than a site. **Advanced** holds every RFdiffusion setting there is, and
   `overrides` takes anything it does not list. Each finished job has a
   **Command** button showing the exact `run_inference.py` line it ran — worth
   a look when a setting does not seem to have taken.
3. Look at them. Then, on the job, press **sequence** under the design you want
   to take further. ProteinMPNN writes sequences for it and the best few are
   folded; the settings are in **Sequence and fold**.
4. The result loads like any other design, superimposed on the backbone it came
   from — so what you see is the predicted protein sitting on your residues.

The number that matters is **best fold ... Å from the backbone**. The sequence
was folded knowing nothing about the shape it was designed for, so how close it
lands is how much the sequence really encodes that shape. Under 2 Å with a
pLDDT above 80 is the usual bar for a design worth making; well above that means
the backbone was a shape nobody can build, and it is cheaper to learn it here.

**Sequences** and **FASTA** on a finished sequence job give you the designs
themselves, with their numbers in the headers.

Rough costs, per backbone: sequence design is seconds; folding is about a minute
each after a first load of a minute or two. Stage 2 on one backbone is therefore
a few minutes, against five to ten for the backbone itself.

## If the app says the endpoint is unreachable

Run `check_link()` first — it tests the same path the app uses and separates the
two cases in one step.

**"no longer resolves"** — the tunnel has closed. Colab disconnected, the
session was recycled, or cloudflared stopped. Run `start_tunnel()`, then restart
proteinCAD with the new URL *and* token. Quick tunnel URLs never survive a
session.

**`530` / `502` / "nothing is serving behind it"** — the tunnel is alive but the
worker is not. Run the worker cell; it clears the port first, so re-running it
is safe.

**"port is already in use"** — an earlier worker is still holding it. The worker
cell handles this now, but `stop_worker()` does it on demand, and
`!fuser -k 8000/tcp` is the blunt version.

## When a design fails

The app shows the worker's own words under the failed job, and the worker adds
what it knows about this machine. Read it in three parts:

* **`Likely cause:`** — read off RFdiffusion's output for this run.
* **`--- environment ---`** — standing facts: the checkout, the weights, the GPU,
  anything that will not import. Re-checked fresh for every job, so a fix in
  another cell shows up on the next run.
* **`--- last output ---`** — RFdiffusion's final lines, verbatim.

Almost everything in the environment section is fixed by re-running the setup
cell. It is idempotent, so there is no cost to trying it. Specifically:

**`none of RFdiffusion's dependencies are importable`** — the setup cell has not
run, or ran in a different runtime. Run it.

**`cannot import dgl`** — the usual one. If the message mentions `collections`
you have DGL 0.4 from PyPI; if it mentions `numpy` or `_ARRAY_API`, DGL is built
against numpy 1 and numpy 2 is installed (`!pip install "numpy<2"`). The setup
cell handles both, the second one automatically.

**`Weights only load failed` / `UnpicklingError`** — torch 2.6 or newer. The
setup pins a torch version that still reads these checkpoints.

**`no CUDA device`** — Runtime → Change runtime type → GPU. Then re-run from the
tunnel cell, because changing the runtime restarts everything.

**A rejected config override** (`Could not override`, `is not in struct`) — this
build of RFdiffusion does not have that key. Restart the worker with
`--no-binder-defaults` added to the `start_worker(...)` call.

**`the generator produced nothing`** — RFdiffusion exited cleanly but wrote no
backbones. Check the contig string in the command printed above the run, or in
the **Command** button on the job in the app.

**`these settings would be rejected`** — caught before the job starts rather
than a minute into it. The line names the setting and why; fix it in
**Advanced** and run again.

**A setting seems to have been ignored** — press **Command** on the job. If the
override is not in that line, this worker is running an older `colab_worker.py`
than the app: re-upload it and re-run the worker cell. The tunnel and token
survive that; only the process restarts.

Nothing here needs the tunnel restarted. A pip install in any cell takes effect
on the next design, because each job spawns a fresh `run_inference.py` — the URL
and the token stay valid.

## Notes worth keeping in mind

**RFdiffusion produces backbones, not proteins.** There is no sequence in the
output. A usable pipeline is RFdiffusion → **ProteinMPNN** (sequence) →
**AlphaFold2 initial-guess or ESMFold** (does it fold as designed?), then filter
on interface pAE and RMSD to the design. Without those last two stages you are
looking at shapes, not candidates.

**Send a small target.** The app already crops around the picked site — keep the
crop radius modest so the target stays in the low hundreds of residues.

**Expect to generate many and keep few.** Tens to hundreds of backbones per
site, filtered hard, is the normal working ratio.

**The tunnel URL changes every session.** That is fine for experiments. For
anything standing, run the same `colab_worker.py` on an EC2 instance behind a
stable address and set `PROTEINCAD_COMPUTE_URL` once.

**`operator torchvision::nms does not exist`** — nothing here uses torchvision.
It is Colab's, it was compiled against Colab's torch, and pinning torch for DGL
stranded it; `import transformers` then trips over it while loading ESMFold, so
the error names an image library and the failure is a protein folder. Re-run the
setup cell — it checks torch's compiled siblings and reinstalls the ones the
pin left behind.

**`Could not import module 'EsmForProteinFolding'`** — that sentence is
transformers' lazy loader catching the real error and replacing it with a
summary. The worker unwraps it, so the message under a failed job names the
actual cause; if you see the bare sentence anywhere else, run
`!"$PY" "$WORKER" setup --only fold` to get the real one.

**`the predictor was killed ... ran out of system memory`** — the folding weights
are about 8.4 GB and a plain Colab session has roughly twelve of RAM, most of
which is already spoken for. The worker now loads the model in half precision
and brings everything but the language stem back up when it sees less than 14 GB
free, and it stops transformers dragging TensorFlow in alongside torch (two
gigabytes this process has no use for — if you saw TensorFlow in the output,
that was it). If it is still killed, **Runtime → Change runtime type → High-RAM**
is the reliable way past it.

**`CUDA out of memory`** — the *card*, not system memory, so a high-RAM runtime
does not help. Read the numbers in the message: if it names another process
holding several gigabytes, that is the problem. The worker now runs one job at a
time and refuses to start when the card is already full, saying what is holding
it. To see for yourself:

```
!nvidia-smi
```

If a fold from an abandoned job is holding it, kill that rather than the
session — restarting the session costs the tunnel, the worker and the URL the
app is pointed at, and none of those are the problem:

```
!pkill -f proteincad_fold_run.py
```

The worker now does this itself before every fold, so it should not recur.
Folding needs roughly 6 GB free.

If you do have to restart the session, the app no longer has to be restarted
with it: open **Design → Compute endpoint**, paste the new tunnel URL and token,
and press **Connect**.
